In [2]:
import wikipediaapi
import requests
from bs4 import BeautifulSoup
import json

def get_citations_via_soup(url):
    """Bóc tách chính xác danh sách chú thích bằng BeautifulSoup"""
    try:
        header = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=header)
        soup = BeautifulSoup(response.content, 'html.parser')
        citations = []
        
        # Tìm danh sách chú thích trong các class phổ biến của Wiki
        ref_list = soup.find('ol', {'class': 'references'})
        if not ref_list:
            ref_list = soup.find('div', {'class': 'reflist'})
            
        if ref_list:
            for li in ref_list.find_all('li'):
                # Lấy text và làm sạch dấu điều hướng ^
                text = li.get_text(separator=" ").strip()
                clean_text = text.lstrip('^ ').strip()
                citations.append(clean_text)
        return citations
    except Exception as e:
        return [f"Lỗi truy xuất HTML: {str(e)}"]

def get_sections_recursive(sections):
    """Hàm đệ quy bảo toàn cấu trúc phân cấp nội dung"""
    structure = []
    for s in sections:
        structure.append({
            "title": s.title,
            "text": s.text.strip(),
            "subsections": get_sections_recursive(s.sections)
        })
    return structure

def scrape_vua_viet_nam():
    # Khai báo API (ngôn ngữ tiếng Việt)
    wiki = wikipediaapi.Wikipedia(
        user_agent='HistoricalResearchBot/1.0',
        language='vi',
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    page_name = "Vua Việt Nam"
    print(f"Đang thu thập dữ liệu từ: {page_name}...")
    page = wiki.page(page_name)

    if not page.exists():
        print("Trang không tồn tại!")
        return

    # 1. Lấy cấu trúc nội dung phân cấp
    hierarchy = get_sections_recursive(page.sections)

    # 2. Lấy danh sách chú thích (Citations)
    citations = get_citations_via_soup(page.fullurl)

    # 3. Cấu trúc hóa thành JSON
    data = {
        "page_title": page.title,
        "url": page.fullurl,
        "summary": page.summary.strip(),
        "content_hierarchy": hierarchy,
        "all_text_clean": page.text.strip(),
        "citations": citations
    }

    # Xuất ra file JSON
    output_file = "vua_viet_nam_data.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Xong! Đã lưu dữ liệu vào '{output_file}'")
    print(f"Số lượng chú thích lấy được: {len(citations)}")

if __name__ == "__main__":
    scrape_vua_viet_nam()

Đang thu thập dữ liệu từ: Vua Việt Nam...


KeyboardInterrupt: 

In [3]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json
import time
import re

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def clean_text(text):
    """Xóa chú thích [1], [2] và khoảng trắng thừa"""
    text = re.sub(r'\[\d+\]', '', text)
    return ' '.join(text.split())

def get_king_list():
    """Bước 1: Lấy danh sách tên vua từ trang tổng hợp"""
    url = "https://vi.wikipedia.org/wiki/Vua_Vi%E1%BB%87t_Nam"
    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    kings = set()
    # Tìm tất cả các bảng danh sách vua
    tables = soup.find_all('table', class_='wikitable')
    
    for table in tables:
        rows = table.find_all('tr')[1:] # Bỏ hàng tiêu đề
        for row in rows:
            cols = row.find_all('td')
            if cols:
                # Thông thường tên vua nằm ở cột 1 hoặc cột 2 tùy bảng
                # Lấy thẻ <a> đầu tiên trong các cột đầu
                for col in cols[:2]:
                    link = col.find('a')
                    if link and not link.get('href').startswith('#'):
                        name = link.get_text().strip()
                        if name and "Nhà" not in name:
                            kings.add(name)
    
    king_list = sorted(list(kings))
    with open('danh_sach_vua.txt', 'w', encoding='utf-8') as f:
        for king in king_list:
            f.write(king + '\n')
    return king_list

def get_king_details_systematic(king_name):
    """Bước 2: Tải và chuẩn hóa JSON từng vị vua theo hệ thống mục lục"""
    base_url = "https://vi.wikipedia.org/wiki/"
    url = base_url + king_name.replace(" ", "_")
    
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code != 200: return None
        
        soup = BeautifulSoup(res.text, 'html.parser')
        king_data = {
            "nhan_vat": king_name,
            "url_nguon": url,
            "tieu_su_he_thong": []
        }

        # Tìm vùng nội dung chính
        content_div = soup.find('div', {'id': 'mw-content-text'})
        if content_div:
            current_section = "Giới thiệu"
            section_content = []
            
            # Quét tuần tự các thẻ p, h2, h3 để đảm bảo lấy hết chữ
            for elem in content_div.find_all(['h2', 'h3', 'p']):
                if elem.name in ['h2', 'h3']:
                    if section_content:
                        king_data["tieu_su_he_thong"].append({
                            "muc": current_section,
                            "noi_dung": clean_text(" ".join(section_content))
                        })
                    current_section = elem.get_text().replace('[sửa | sửa mã nguồn]', '').strip()
                    section_content = []
                elif elem.name == 'p':
                    text = elem.get_text().strip()
                    if text: section_content.append(text)
            
            # Lưu mục cuối
            if section_content:
                king_data["tieu_su_he_thong"].append({
                    "muc": current_section,
                    "noi_dung": clean_text(" ".join(section_content))
                })
        return king_data
    except:
        return None

# --- Chạy quy trình ---
kings = get_king_list()
print(f"Đã lấy được {len(kings)} vị vua. Bắt đầu tải chi tiết...")

all_kings_json = []
for name in kings[:10]: # Thử nghiệm với 10 người đầu tiên
    data = get_king_details_systematic(name)
    if data:
        all_kings_json.append(data)
    time.sleep(0.5)

with open('vua_viet_nam_chuan_hoa.json', 'w', encoding='utf-8') as f:
    json.dump(all_kings_json, f, ensure_ascii=False, indent=4)

Đã lấy được 70 vị vua. Bắt đầu tải chi tiết...


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re

def clean_text(text):
    """Làm sạch các ký tự đặc biệt, chú thích [1], [a] và khoảng trắng thừa"""
    if not text: return ""
    # Loại bỏ các tham chiếu dạng [1], [a], [ghi chú 1]
    text = re.sub(r'\[[^\]]+\]', '', text)
    # Loại bỏ dấu ngoặc đơn và nội dung bên trong nếu chỉ chứa số (năm sinh/mất)
    text = re.sub(r'\(\d+–\d+\)', '', text)
    return text.strip()

def get_vua_viet_nam_data(url):
    header = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    try:
        response = requests.get(url, headers=header)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.content, 'html.parser')
        
        all_kings = []
        kings_metadata = {}

        # Tìm tất cả các bảng wikitable trên trang
        tables = soup.find_all('table', {'class': 'wikitable'})
        print(f"Tìm thấy {len(tables)} bảng dữ liệu.")

        for index, table in enumerate(tables):
            # Cố gắng xác định triều đại dựa trên tiêu đề (h3, h2) gần nhất phía trên bảng
            prev_node = table.find_previous(['h3', 'h2', 'h4'])
            dynasty_name = prev_node.get_text().replace('[ sửa | sửa mã nguồn ]', '').strip() if prev_node else f"Nhóm {index+1}"
            
            rows = table.find_all('tr')
            if not rows: continue
            
            kings_in_dynasty = []
            
            # Xác định vị trí cột tên (thường là cột 1 hoặc 2)
            # Dựa vào tiêu đề bảng (th)
            header_cols = [th.get_text().strip() for th in rows[0].find_all(['th', 'td'])]
            name_idx = -1
            for i, col in enumerate(header_cols):
                if any(kw in col for kw in ["Vua", "Chúa", "Vương hiệu", "Hoàng đế", "Tên"]):
                    name_idx = i
                    break
            
            # Nếu không tìm thấy header rõ ràng, mặc định thử cột 1 hoặc 2
            target_idx = name_idx if name_idx != -1 else 1

            for row in rows[1:]:
                cols = row.find_all(['td', 'th'])
                if len(cols) > target_idx:
                    name_cell = cols[target_idx]
                    
                    # Lấy text sạch
                    raw_name = name_cell.get_text(separator=" ").strip()
                    name = clean_text(raw_name)
                    
                    # Bỏ qua nếu là các hàng tiêu đề phụ hoặc rỗng
                    if not name or len(name) < 2 or "đời vua" in name.lower():
                        continue
                    
                    # Lấy thông tin Hán Nôm (thường ở cột ngay sau tên)
                    han_nom = ""
                    if len(cols) > target_idx + 1:
                        han_nom = clean_text(cols[target_idx + 1].get_text())

                    king_entry = {
                        "vương_hiệu": name,
                        "hán_nôm": han_nom,
                        "triều_đại": dynasty_name
                    }
                    kings_in_dynasty.append(king_entry)
                    all_kings.append(name)

            if kings_in_dynasty:
                kings_metadata[dynasty_name] = kings_in_dynasty

        return kings_metadata, list(dict.fromkeys(all_kings)) # Trả về metadata và danh sách không trùng

    except Exception as e:
        print(f"Lỗi: {e}")
        return {}, []

def main():
    url = "https://vi.wikipedia.org/wiki/Vua_Vi%E1%BB%87t_Nam"
    print("Đang quét dữ liệu...")
    
    metadata, flat_list = get_vua_viet_nam_data(url)
    
    # 1. Lưu file JSON cấu trúc
    with open("vua_viet_nam_structured.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=4)
        
    # 2. Lưu file TXT phẳng để bạn xử lý tiếp
    with open("danh_sach_vua.txt", "w", encoding="utf-8") as f:
        for name in flat_list:
            # Lọc bỏ thêm các dòng tiêu đề trùng lặp nếu có
            if name not in ["Vua", "Vương hiệu", "Tên"]:
                f.write(name + "\n")

    print(f"--- KẾT QUẢ ---")
    print(f"Tổng số vị vua/chúa tìm thấy: {len(flat_list)}")
    print(f"Đã lưu vào 'vua_viet_nam_structured.json' và 'danh_sach_vua.txt'")

if __name__ == "__main__":
    main()

Đang quét dữ liệu...
Tìm thấy 8 bảng dữ liệu.
--- KẾT QUẢ ---
Tổng số vị vua/chúa tìm thấy: 72
Đã lưu vào 'vua_viet_nam_structured.json' và 'danh_sach_vua.txt'


: 

In [14]:
import requests
from bs4 import BeautifulSoup
import json
import time
import re

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def clean_text(text):
    """Xóa chú thích [1], [2] và khoảng trắng thừa"""
    text = re.sub(r'\[.*?\]', '', text)
    return ' '.join(text.split()).strip()

def get_king_list():
    """Bước 1: Lấy danh sách tên nhân vật (Vua, Chúa, Tiết độ sứ...)"""
    url = "https://vi.wikipedia.org/wiki/Vua_Vi%E1%BB%87t_Nam"
    response = requests.get(url, headers=HEADERS)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, 'html.parser')
    
    kings_dict = {} # Dùng dict để lưu {Tên: URL} tránh trùng lặp
    
    # Tìm tất cả các bảng dữ liệu
    tables = soup.find_all('table', class_='wikitable')
    
    for table in tables:
        rows = table.find_all('tr')
        for row in rows:
            # Lấy tất cả các ô trong hàng (không giới hạn cột)
            cells = row.find_all(['td', 'th'])
            for cell in cells:
                # Tìm tất cả các link trong ô
                links = cell.find_all('a')
                for link in links:
                    href = link.get('href', '')
                    name = link.get_text().strip()
                    title = link.get('title', '')

                    # ĐIỀU KIỆN LỌC QUAN TRỌNG:
                    # 1. Link phải dẫn đến bài viết (/wiki/...)
                    # 2. Không phải link nội bộ (#), link tập tin, hay link thảo luận
                    # 3. Không chứa từ "Nhà" (ví dụ Nhà Lý), "Kỷ", "Bắc thuộc"
                    if href.startswith('/wiki/') and ':' not in href:
                        if not any(x in name for x in ["Nhà", "Kỷ", "Bắc thuộc", "sửa mã nguồn"]):
                            if name and len(name) > 1:
                                # Ưu tiên lấy title của link vì nó chứa tên đầy đủ sạch nhất
                                clean_name = re.sub(r' \(.*?\)', '', title) if title else name
                                kings_dict[clean_name] = "https://vi.wikipedia.org" + href

    # Chuyển về list và sắp xếp
    sorted_kings = sorted([{"name": k, "url": v} for k, v in kings_dict.items()], key=lambda x: x['name'])
    
    with open('danh_sach_vua.txt', 'w', encoding='utf-8') as f:
        for item in sorted_kings:
            f.write(item['name'] + '\n')
            
    return sorted_kings

def get_king_details_systematic(king_name, url):
    """Bước 2: Tải và chuẩn hóa JSON từng nhân vật"""
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code != 200: return None
        
        soup = BeautifulSoup(res.text, 'html.parser')
        king_data = {
            "nhan_vat": king_name,
            "url_nguon": url,
            "tieu_su_he_thong": []
        }

        content_div = soup.find('div', class_='mw-parser-output')
        if content_div:
            # Xóa bỏ các phần không cần thiết
            for junk in content_div.find_all(['table', 'div'], class_=['infobox', 'navbox', 'reflist', 'wmpp-notice']):
                junk.decompose()

            current_section = "Giới thiệu"
            section_content = []
            
            for elem in content_div.find_all(['h2', 'h3', 'p', 'ol'], recursive=False):
                if elem.name in ['h2', 'h3']:
                    if section_content:
                        king_data["tieu_su_he_thong"].append({
                            "muc": current_section,
                            "noi_dung": clean_text("\n".join(section_content))
                        })
                    current_section = elem.get_text().replace('[sửa | sửa mã nguồn]', '').strip()
                    # Thoát khi gặp các mục lục cuối trang
                    if any(x in current_section for x in ["Tham khảo", "Ghi chú", "Liên kết ngoài"]):
                        section_content = []
                        break
                    section_content = []
                elif elem.name in ['p', 'ol']:
                    text = elem.get_text(separator=" ").strip()
                    if text: section_content.append(text)
            
            if section_content:
                king_data["tieu_su_he_thong"].append({
                    "muc": current_section,
                    "noi_dung": clean_text("\n".join(section_content))
                })
        return king_data
    except:
        return None

# --- CHẠY CHƯƠNG TRÌNH ---
if __name__ == "__main__":
    print("--- Bước 1: Thu thập danh sách nhân vật (Vua, Chúa, Tiết độ sứ...) ---")
    king_list = get_king_list()
    total = len(king_list)
    print(f"Tìm thấy tổng cộng {total} nhân vật (Số lượng này chắc chắn > 71).")

    print("\n--- Bước 2: Tải nội dung chi tiết (Thử nghiệm 20 người đầu) ---")
    all_data = []
    # Bạn có thể bỏ [:20] để quét toàn bộ danh sách
    for i, item in enumerate(king_list[:20]):
        print(f"[{i+1}/{total}] Đang tải: {item['name']}")
        details = get_king_details_systematic(item['name'], item['url'])
        if details:
            all_data.append(details)
        time.sleep(0.5)

    with open('vua_viet_nam_full_data.json', 'w', encoding='utf-8') as f:
        json.dump(all_data, f, ensure_ascii=False, indent=4)
    print(f"\nĐã hoàn thành! Kết quả lưu tại 'vua_viet_nam_full_data.json'")

--- Bước 1: Thu thập danh sách nhân vật (Vua, Chúa, Tiết độ sứ...) ---
Tìm thấy tổng cộng 146 nhân vật (Số lượng này chắc chắn > 71).

--- Bước 2: Tải nội dung chi tiết (Thử nghiệm 20 người đầu) ---
[1/146] Đang tải: An Nhơn
[2/146] Đang tải: Bạch Hạc
[3/146] Đang tải: Bắc Ninh
[4/146] Đang tải: Bồn địa Tứ Xuyên
[5/146] Đang tải: Bộ Ngưu
[6/146] Đang tải: Bộ Nhật
[7/146] Đang tải: Cao Đế
[8/146] Đang tải: Chiêu Tông
[9/146] Đang tải: Chính Định
[10/146] Đang tải: Chúa Nguyễn
[11/146] Đang tải: Chúa Trịnh
[12/146] Đang tải: Cung Đế
[13/146] Đang tải: Cung điện Phiên Ngung
[14/146] Đang tải: Cố đô Hoa Lư
[15/146] Đang tải: Cố đô Huế
[16/146] Đang tải: Duệ Đế
[17/146] Đang tải: Dương Tam Kha
[18/146] Đang tải: Dương Đình Nghệ
[19/146] Đang tải: Gia Lai
[20/146] Đang tải: Hai Bà Trưng

Đã hoàn thành! Kết quả lưu tại 'vua_viet_nam_full_data.json'
